In [1]:
# Setup -------------------------------------------------------------------
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
import sklearn.metrics #import mean_absolute_error, mean_squared_error, r2_score

In [2]:
# Load the mtcars dataset
mtcars = pd.read_csv('../../03-Database/mtcars.csv')

print(mtcars.head())
# Fit the linear regression model
# The R code uses `lm`, which corresponds to `statsmodels.api.OLS` for a similar summary
# For direct prediction, `sklearn.linear_model.LinearRegression` is also a great option
# We'll use statsmodels to get a similar summary to R
X = mtcars[['wt', 'qsec', 'am']]
y = mtcars['mpg']
X = sm.add_constant(X)  # Add a constant for the intercept
ols_mdl = sm.OLS(y, X).fit()


    mpg  cyl   disp   hp  drat     wt   qsec  vs  am  gear  carb
0  21.0    6  160.0  110  3.90  2.620  16.46   0   1     4     4
1  21.0    6  160.0  110  3.90  2.875  17.02   0   1     4     4
2  22.8    4  108.0   93  3.85  2.320  18.61   1   1     4     1
3  21.4    6  258.0  110  3.08  3.215  19.44   1   0     3     1
4  18.7    8  360.0  175  3.15  3.440  17.02   0   0     3     2


In [3]:
# The R `summary(ols.mdl)` is equivalent to the following
print("--- Model Summary (statsmodels) ---")
print(ols_mdl.summary())
print("-----------------------------------")


--- Model Summary (statsmodels) ---
                            OLS Regression Results                            
Dep. Variable:                    mpg   R-squared:                       0.850
Model:                            OLS   Adj. R-squared:                  0.834
Method:                 Least Squares   F-statistic:                     52.75
Date:                Thu, 11 Sep 2025   Prob (F-statistic):           1.21e-11
Time:                        20:20:16   Log-Likelihood:                -72.060
No. Observations:                  32   AIC:                             152.1
Df Residuals:                      28   BIC:                             158.0
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          9

In [4]:
# Prediction ----------------------------------------------------------------

# The R `predict(ols.mdl)` is equivalent to `ols_mdl.predict(X)`
y_hat = ols_mdl.predict(X)

# Create a DataFrame similar to the R `tbl`
tbl = pd.DataFrame({
    'mpg': mtcars['mpg'],
    'mpg_hat': y_hat,
    'e': mtcars['mpg'] - y_hat
})

print("\n--- Predictions and Residuals ---")
print(tbl.head())
print("---------------------------------")


--- Predictions and Residuals ---
    mpg    mpg_hat         e
0  21.0  22.470461 -1.470461
1  21.0  22.158249 -1.158249
2  22.8  26.281067 -3.481067
3  21.4  20.857444  0.542556
4  18.7  17.009587  1.690413
---------------------------------


In [5]:
# MAE & MSE & RMSE --------------------------------------------------------

# The R code manually calculates these, but Python has built-in functions
mae = sklearn.metrics.mean_absolute_error(tbl['mpg'], tbl['mpg_hat'])
mse = sklearn.metrics.mean_squared_error(tbl['mpg'], tbl['mpg_hat'])
rmse = sklearn.metrics.root_mean_squared_error(tbl['mpg'], tbl['mpg_hat'])
# rmse = np.sqrt(mse)

print("\n--- Model Evaluation Metrics ---")
print(f"MAE: {mae:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")



--- Model Evaluation Metrics ---
MAE: 1.9320
MSE: 5.2902
RMSE: 2.3000


In [6]:
# MAPE --------------------------------------------------------------------

# MAPE must be calculated manually, similar to the R code
mape = np.mean(np.abs(tbl['e'] / tbl['mpg'])) * 100

print(f"MAPE: {mape:.4f}%")

MAPE: 9.5499%


In [7]:
# R-squared & Adjusted R-squared ------------------------------------------

# R-squared can be calculated manually or with a built-in function
# The R code's manual calculation is:
r2_manual = 1 - np.sum(tbl['e']**2) / np.sum((tbl['mpg'] - np.mean(tbl['mpg']))**2)
r2_sklearn = sklearn.metrics.r2_score(tbl['mpg'], tbl['mpg_hat'])


In [8]:

# The adjusted R-squared calculation from R is:
n = len(tbl)
p = ols_mdl.df_model # Number of predictors (wt, qsec, am)
r2_adj = 1 - (1 - r2_manual) * (n - 1) / (n - p - 1)

print(f"R-squared: {r2_sklearn:.4f}")
print(f"Adjusted R-squared: {r2_adj:.4f}")
print("-----------------------------------\n")

# Note: The manual R2 and R2_adj calculations in the R code are a common way to calculate these
# metrics, but the `ols_mdl.summary()` from `statsmodels` provides them automatically.

R-squared: 0.8497
Adjusted R-squared: 0.8336
-----------------------------------

